In [1]:
!pip install pyngrok
!pip install google-adk

In [2]:
# making our agent as a package
!mkdir multi_tool_agent/

In [3]:
# folder becomes a package
!echo "from . import agent" > multi_tool_agent/__init__.py

In [4]:
# check the package
!cat multi_tool_agent/__init__.py

from . import agent


In [5]:
#create agent.py ,it creates a new python module inside this package
!touch multi_tool_agent/agent.py

In [6]:
!adk create multi_tool_agent


Help improve the ADK (CLI and Web UI) by allowing Google to collect pseudonymized usage data?

What is collected: Names of subcommands and flags (no user-provided values or arguments), execution metrics (duration, exit state), environment specs (OS, Python version), and aggregated Web UI feature interactions. No personally identifiable information (PII) is collected.

This is OFF by default. You can opt out at any time using the 'adk telemetry disable' command or Web UI user settings.

Enable telemetry? [Y/n]: Y
Non-empty folder already exist: '/content/multi_tool_agent'
Override existing content? [y/N]: Y
Choose a model for the root agent:
1. gemini-3.5-flash
2. Other models (fill later)
Choose model (1, 2): 1
1. Google AI
2. Vertex AI
3. Login with Google
Choose a backend (1, 2, 3): 1

Don't have API Key? Create one in AI Studio: https://aistudio.google.com/apikey

Enter Google API key: [redacted]

Agent created in /content/multi_tool_agent:
- .env
- .gitignore
- __init__.py
- agent.

In [9]:
import pandas as pd
from google.adk.agents.llm_agent import Agent
from typing import List

try:
  _df = pd.read_csv('sample_data/smartphones.csv')
  _df = _df.dropna(subset = ['model','price','rating','processor','battery','ram','sim'])
except FileNotFoundError:
  print('Error:File smartphones.csv is not found.')
  _df = pd.DataFrame()

def review_smartphone(model_name:str)->str:
  """
  Provides the detailed review and summary of the key specifications of the smart phones.
  """
  if _df.empty:
    return "Error: Smartphone data not loaded."

  data =_df[_df['model'].str.contains(model_name,case=False,na=False)]

  if data.empty:
    return f"Model {model_name} is not in the database."

  data = data.iloc[0]
  review = (f"**Review about {model_name} \n"
  f"Price: ${data['price']}\n"
  f"Average rating: {data['rating']}/10 \n"
  f"Processor :{data['processor']}\n")

  return review

def compare_smartphones(model_1:str,model_2:str)->str:
  """
  Compares the key specifications of two smart phones.
  """
  if _df.empty:
    return "Error: Smartphone data not loaded."

  model_1_data = _df[_df['model'].str.contains(model_1,case=False,na=False)]
  model_2_data = _df[_df['model'].str.contains(model_2,case=False,na=False)]

  if model_1_data.empty or model_2_data.empty:
    return f"Model {model_1} or {model_2} is not in the database."

  data_1 = model_1_data.iloc[0]
  data_2 = model_2_data.iloc[0]

  comparison = (f"**Comparison between {model_1} and {model_2}\n"
  f"Price: ${data_1['price']} vs ${data_2['price']}\n"
  f"Average rating: {data_1['rating']} vs {data_2['rating']}/1 \n"
  f"Processor: {data_1['processor']} vs {data_2['processor']}\n"
  f"Battery: {data_1['battery']} vs {data_2['battery']}\n"
  f"RAM: {data_1['ram']} vs {data_2['ram']}\n"
  f"Sim Type: {data_1['sim']} vs {data_2['sim']}")

  return comparison

def recommend_smartphones(max_price:int,min_rating:float,required_5g:bool)->List[str]:
  """
  Recommends a list of upto five smart phones based on the maximum price , minimum rating and 5G support.
  """
  if _df.empty:
    return "Error: Smartphone data not loaded."

  #convert the 5g required to 0 or 1
  required_5g = 1 if required_5g else 0

  #filter the dataframe
  recommendation_df = _df[( _df['price'] <= max_price) &
                    (_df['rating'] >= min_rating) &
                     (_df['sim'] == required_5g)]
  #sort by rating(highest first) and price (lowest first)
  recommendation_df=recommendation_df.sort_values(by=['rating','price'],ascending=[False,True])

  #get the top 5
  top_models = recommendation_df.head(5)[['model','price','rating'].to_dict('records')]

  if not top_models:
    return "No matching smartphones found."

  result_list = ["Based on the preferences here are the top models recommended: "]
  for models in top_models:
    result_list.append(f"Model: {models['model']}, Price: ${models['price']}, Rating: {models['rating']}")

  return result_list

  root_agent = Agent(name = "smartphone_data_analyst ",
                     model = "gemini-3.5-flash",
                     description = "A multi tool agent for querying and analyzing smartphone specifications from the loaded dataset",
                     instruction =("Yoy are an expert smart phone recommendation analyst./n "
                     "Use only the provided tools to respond to the user queries related to smartphones specifications,"
                     "reviews,comparisons,pricing, ratings, and 5g or other technical capablities. \n"
                     "when a query requires factual smartphone data or structured analysis ,invoke the approppriate tool"
                     "(review_smartphone,compare_smartphones,recommend_smartphones). \n"
                     "For all other queries , or general or non-smartphone related questions answer it with your own reasoning without using any tools. /n"
                     "Ensure your response remains accurate , concise and tailored to the user's needs."),
                     tools = [review_smartphone,
                              compare_smartphones,
                              recommend_smartphones],
                     )

In [10]:
!touch multi_tool_agent/.env

In [11]:
!adk run multi_tool_agent

Log setup complete: /tmp/agents_log/agent.20260830_124234.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.13/dist-packages/google/adk/cli/cli.py:335: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.13/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent root_agent, type exit to exit.
[user]: give me phone suggestions under 15000 rupees
[root_agent]: The under-₹15,000 smartphone segment in India is highly competitive, offering excellent features like 5G connectivity, high ref

**Clean the Enviornment**

In [13]:
!pkill ngrok
!pkill streamlit
!pkill uvicorn

In [14]:
!pip install fastapi uvicorn pyngrok google-adk streamlit plotly requests python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.4 MB/s eta 0:00:00


In [15]:
!mkdir -p /content/smartphone_app/backend
!mkdir -p /content/smartphone_app/frontend


In [16]:
!mkdir -p /content/smartphone_app/Colablauncher

In [17]:
import os
os.environ['GOOGLE_API_KEY'] = os.environ.get('GOOGLE_API_KEY', '')

In [ ]:
%%writefile /content/smartphone_app/backend/main.py
from fastapi import FastAPI,UploadFile,File,HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List,Optional
import pandas as pd
import numpy as np
import io,os ,uvicorn

app = FastAPI(title = "Smartphone API")